# CelebAMask-HQ GPU Training BG / HAIR / FACE 5k 200-Step Probe

Use this notebook with a GPU runtime after the BG/HAIR/FACE CPU notebook has
already created the train/val shard package. This run intentionally keeps the
same stable alpha-aware micro-ablation recipe but extends training from 100 to
200 steps, so we can test whether longer training helps or hurts the latest
BG/HAIR/FACE setup.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/CelebMaskHQ_Colab')
REPO_DRIVE = DRIVE_ROOT / 'repo'
PROCESSED_DRIVE_ROOT = DRIVE_ROOT / 'processed'
RUNS_DRIVE_ROOT = DRIVE_ROOT / 'runs'

PROJECT_LOCAL = Path('/content/project')
SHARDS_LOCAL_ROOT = Path('/content/trainval_shards')

SHARD_OUTPUT_NAME = 'processed_celebmaskhq_bghairface_5k_trainval_shards'
SHARD_OUTPUT_DRIVE = PROCESSED_DRIVE_ROOT / SHARD_OUTPUT_NAME
PACKAGE_MANIFEST_DRIVE = SHARD_OUTPUT_DRIVE / 'package_manifest.json'

RUN_NAME = 'qwen_layered_lora_bghairface_5k_200step_probe'
RUN_DRIVE = RUNS_DRIVE_ROOT / RUN_NAME
RUN_LOCAL = Path(f'/content/{RUN_NAME}')

RESOLUTION = 640
MAX_LAYERS = 3
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
FULL_MAX_STEPS = 200
CHECKPOINTING_STEPS = 25
VALIDATION_STEPS = 25
MAX_VALIDATION_SAMPLES = 16
NUM_VALIDATION_BATCHES = 2
SAVE_TOTAL_LIMIT = 6
LEARNING_RATE = 1e-5
RANK = 8
LORA_ALPHA = 8
MIXED_PRECISION = 'bf16'

FORCE_RESTART_FULL_RUN = False
RESUME_IF_AVAILABLE = True
SKIP_VALIDATION = False
ALLOW_TF32 = True

print('Repo on Drive:', REPO_DRIVE)
print('Train/val shard package on Drive:', SHARD_OUTPUT_DRIVE)
print('Package manifest on Drive:', PACKAGE_MANIFEST_DRIVE)
print('Local shard staging root:', SHARDS_LOCAL_ROOT)
print('Mirrored training run on Drive:', RUN_DRIVE)
print('200-step probe defaults: lr=1e-5, rank=8, alpha=8, grad_accum=4, max_steps=200, checkpoint_every=25')
print('This 200-step probe uses a fresh RUN_NAME so prior micro-ablation checkpoints are not reused.')


In [ ]:
!nvidia-smi


In [ ]:
!pip uninstall -y torchao
!pip install -U accelerate peft bitsandbytes sentencepiece protobuf
!pip install -U "transformers>=4.51.3"
!pip install git+https://github.com/huggingface/diffusers.git


## If Colab asks for a runtime restart

Restart the runtime, then rerun the notebook from the top.


In [ ]:
import json
import shutil
import subprocess

assert REPO_DRIVE.exists(), f'Missing repo folder on Drive: {REPO_DRIVE}'
assert PACKAGE_MANIFEST_DRIVE.exists(), f'Missing package manifest on Drive: {PACKAGE_MANIFEST_DRIVE}'

package_manifest = json.loads(PACKAGE_MANIFEST_DRIVE.read_text(encoding='utf-8'))
assert package_manifest['included_splits'] == ['train', 'val'], package_manifest['included_splits']
assert package_manifest['test_split_packaged'] == 0, package_manifest['test_split_packaged']

PACKAGE_NAME = package_manifest['package_name']
METADATA_TAR_DRIVE = SHARD_OUTPUT_DRIVE / package_manifest['metadata_tar']
DATA_SHARD_DRIVE_PATHS = [SHARD_OUTPUT_DRIVE / shard['filename'] for shard in package_manifest['data_shards']]
PROCESSED_LOCAL = Path('/content') / PACKAGE_NAME
METADATA_TAR_LOCAL = SHARDS_LOCAL_ROOT / package_manifest['metadata_tar']
DATA_SHARD_LOCAL_PATHS = [SHARDS_LOCAL_ROOT / shard['filename'] for shard in package_manifest['data_shards']]

if PROJECT_LOCAL.exists():
    shutil.rmtree(PROJECT_LOCAL)
if SHARDS_LOCAL_ROOT.exists():
    shutil.rmtree(SHARDS_LOCAL_ROOT)
if PROCESSED_LOCAL.exists():
    shutil.rmtree(PROCESSED_LOCAL)
if RUN_LOCAL.exists():
    shutil.rmtree(RUN_LOCAL)
if FORCE_RESTART_FULL_RUN and RUN_DRIVE.exists():
    shutil.rmtree(RUN_DRIVE)

SHARDS_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(REPO_DRIVE, PROJECT_LOCAL)
shutil.copy2(METADATA_TAR_DRIVE, METADATA_TAR_LOCAL)
for source_path, target_path in zip(DATA_SHARD_DRIVE_PATHS, DATA_SHARD_LOCAL_PATHS, strict=True):
    shutil.copy2(source_path, target_path)

extract_commands = [
    ['tar', '-xf', str(METADATA_TAR_LOCAL), '-C', '/content'],
]
extract_commands.extend([
    ['tar', '-xf', str(shard_path), '-C', '/content']
    for shard_path in DATA_SHARD_LOCAL_PATHS
])
for command in extract_commands:
    print('Running extract command:', ' '.join(command))
    subprocess.run(command, check=True)

assert PROCESSED_LOCAL.exists(), f'Processed package was not extracted: {PROCESSED_LOCAL}'
assert (PROCESSED_LOCAL / 'metadata' / 'layered_samples.jsonl').exists(), (
    f'Missing layered metadata after extract: {PROCESSED_LOCAL / "metadata" / "layered_samples.jsonl"}'
)

RUNS_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RESUME_FROM_CHECKPOINT = None
RESUME_CHECKPOINT_NAMES = []
if RUN_DRIVE.exists() and RESUME_IF_AVAILABLE and not FORCE_RESTART_FULL_RUN:
    shutil.copytree(RUN_DRIVE, RUN_LOCAL)
    RESUME_CHECKPOINT_NAMES = sorted(
        [path.name for path in RUN_LOCAL.iterdir() if path.is_dir() and path.name.startswith('checkpoint-')],
        key=lambda name: int(name.split('-', 1)[1]),
    )
    if RESUME_CHECKPOINT_NAMES:
        RESUME_FROM_CHECKPOINT = 'latest'

print('Package name:', PACKAGE_NAME)
print('Extracted processed package to', PROCESSED_LOCAL)
print('Copied shard count:', len(DATA_SHARD_LOCAL_PATHS))
print('Resume checkpoint names:', RESUME_CHECKPOINT_NAMES)


In [ ]:
import json

layered_manifest_path = PROCESSED_LOCAL / 'metadata' / 'layered_samples.jsonl'
rows = [json.loads(line) for line in layered_manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]
row_splits = sorted(set(row['split'] for row in rows))
assert row_splits == ['train', 'val'], row_splits
assert all(row['split'] != 'test' for row in rows)
assert all(row['layer_names'] == ['BG', 'HAIR', 'FACE'] for row in rows), rows[0]
assert all(row['layer_count'] == 3 for row in rows), rows[0]
print('Extracted manifest split check passed:', row_splits)
print('Extracted sample count:', len(rows))


In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    'scripts/inspect_generic_layered_loader.py',
    '--processed-root', str(PROCESSED_LOCAL),
    '--split', 'train',
    '--batch-size', '1',
    '--num-batches', '1',
    '--max-layers', str(MAX_LAYERS),
    '--drop-warning-samples',
], cwd=PROJECT_LOCAL, check=True)


In [ ]:
trainer_path = PROJECT_LOCAL / 'scripts' / 'train_qwen_image_layered_lora.py'
assert trainer_path.exists(), f'Missing trainer script: {trainer_path}'

trainer_text = trainer_path.read_text(encoding='utf-8')
freshness_markers = [
    '--resume-from-checkpoint',
    '--mirror-output-dir',
    'load_resume_lora_into_transformer',
    'ensure_lora_adapter_trainable',
    'No trainable LoRA parameters were found after adapter setup.',
    'save_checkpoint(',
    'completed_batches',
]
missing_markers = [marker for marker in freshness_markers if marker not in trainer_text]
assert not missing_markers, (
    'The copied Colab trainer looks stale or incomplete. '
    f'Missing markers: {missing_markers}. '
    'Refresh the repo on Drive from the local workspace and rerun from the top.'
)

print('Trainer freshness check passed:', trainer_path)
print('Verified resume markers:', freshness_markers)


In [ ]:
import subprocess
import sys

train_command = [
    sys.executable,
    'scripts/train_qwen_image_layered_lora.py',
    '--processed-root', str(PROCESSED_LOCAL),
    '--output-dir', str(RUN_LOCAL),
    '--mirror-output-dir', str(RUN_DRIVE),
    '--resolution', str(RESOLUTION),
    '--train-batch-size', str(TRAIN_BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRADIENT_ACCUMULATION_STEPS),
    '--num-train-epochs', '1',
    '--max-train-steps', str(FULL_MAX_STEPS),
    '--checkpointing-steps', str(CHECKPOINTING_STEPS),
    '--validation-steps', str(VALIDATION_STEPS),
    '--max-validation-samples', str(MAX_VALIDATION_SAMPLES),
    '--num-validation-batches', str(NUM_VALIDATION_BATCHES),
    '--save-total-limit', str(SAVE_TOTAL_LIMIT),
    '--max-layers', str(MAX_LAYERS),
    '--rank', str(RANK),
    '--lora-alpha', str(LORA_ALPHA),
    '--learning-rate', str(LEARNING_RATE),
    '--mixed-precision', MIXED_PRECISION,
    '--gradient-checkpointing',
    '--drop-warning-samples',
]

if ALLOW_TF32:
    train_command.append('--allow-tf32')
if SKIP_VALIDATION:
    train_command.append('--skip-validation')
if RESUME_FROM_CHECKPOINT is not None:
    train_command.extend(['--resume-from-checkpoint', RESUME_FROM_CHECKPOINT])

print('Running training command:', ' '.join(train_command))
result = subprocess.run(train_command, cwd=PROJECT_LOCAL, text=True, capture_output=True)
print('RETURN CODE:', result.returncode)
print('--- STDOUT ---')
print(result.stdout)
print('--- STDERR ---')
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError('Training failed; see stdout/stderr above.')


In [ ]:
import json

assert RUN_LOCAL.exists(), f'Missing local run directory: {RUN_LOCAL}'
assert RUN_DRIVE.exists(), f'Missing mirrored Drive run directory: {RUN_DRIVE}'

required_relative_files = [
    'run_config.json',
    'trainer_state.json',
    'logs/training_metrics.jsonl',
    'final/final_summary.json',
]

def collect_checkpoint_names(root: Path):
    names = []
    for path in root.iterdir():
        if not path.is_dir() or not path.name.startswith('checkpoint-'):
            continue
        int(path.name.split('-', 1)[1])
        names.append(path.name)
    return sorted(names, key=lambda name: int(name.split('-', 1)[1]))

for root in [RUN_LOCAL, RUN_DRIVE]:
    missing = [relative_path for relative_path in required_relative_files if not (root / relative_path).exists()]
    assert not missing, f'Missing required files under {root}: {missing}'

local_checkpoint_names = collect_checkpoint_names(RUN_LOCAL)
drive_checkpoint_names = collect_checkpoint_names(RUN_DRIVE)
assert local_checkpoint_names, f'No checkpoint-* directories found under {RUN_LOCAL}'
assert drive_checkpoint_names == local_checkpoint_names, (
    f'Checkpoint mismatch between local and Drive runs. Local={local_checkpoint_names} Drive={drive_checkpoint_names}'
)

latest_checkpoint_name = local_checkpoint_names[-1]
latest_checkpoint_dir = RUN_LOCAL / latest_checkpoint_name
assert (latest_checkpoint_dir / 'pytorch_lora_weights.safetensors').exists(), (
    f'Missing LoRA weights in latest checkpoint: {latest_checkpoint_dir}'
)
assert (latest_checkpoint_dir / 'training_state.json').exists(), (
    f'Missing training_state.json in latest checkpoint: {latest_checkpoint_dir}'
)

trainer_state = json.loads((RUN_LOCAL / 'trainer_state.json').read_text(encoding='utf-8'))
final_summary = json.loads((RUN_LOCAL / 'final' / 'final_summary.json').read_text(encoding='utf-8'))

print('Sharded train/val full-run artifact verification passed.')
print('Latest checkpoint:', latest_checkpoint_name)
print('Trainer state:', trainer_state)
print('Final summary:', final_summary)


In [ ]:
import json

log_path = RUN_LOCAL / 'logs' / 'training_metrics.jsonl'
assert log_path.exists(), f'Missing training log: {log_path}'

log_rows = [json.loads(line) for line in log_path.read_text(encoding='utf-8').splitlines() if line.strip()]
train_rows = [row for row in log_rows if 'train_loss' in row]
validation_rows = [row for row in log_rows if 'validation_loss' in row]
assert train_rows, 'No train_loss rows found in training_metrics.jsonl'

checkpoint_steps = sorted(int(name.split('-', 1)[1]) for name in local_checkpoint_names if name.startswith('checkpoint-'))
if not checkpoint_steps:
    checkpoint_steps = sorted(
        int(path.name.split('-', 1)[1])
        for path in RUN_LOCAL.glob('checkpoint-*')
        if path.is_dir() and path.name.split('-', 1)[1].isdigit()
    )
assert checkpoint_steps, f'No checkpoint-* directories found under {RUN_LOCAL}'

def nearest_checkpoint_for_validation_step(step):
    eligible = [checkpoint_step for checkpoint_step in checkpoint_steps if checkpoint_step <= step]
    if eligible:
        return max(eligible)
    return min(checkpoint_steps, key=lambda checkpoint_step: abs(checkpoint_step - step))

if validation_rows:
    ranked_validation_rows = sorted(validation_rows, key=lambda row: row['validation_loss'])
    best_validation_row = ranked_validation_rows[0]
    best_validation_step = int(best_validation_row['step'])
    best_checkpoint_step = nearest_checkpoint_for_validation_step(best_validation_step)
    best_checkpoint_name = f'checkpoint-{best_checkpoint_step}'
    print('Best validation step:', best_validation_step)
    print('Best validation loss:', best_validation_row['validation_loss'])
    print('Suggested checkpoint for inference:', best_checkpoint_name)
    print('Available checkpoint steps:', checkpoint_steps)
    print('Top validation checkpoints:')
    for row in ranked_validation_rows[:5]:
        ranked_step = int(row['step'])
        ranked_checkpoint_step = nearest_checkpoint_for_validation_step(ranked_step)
        print(f"  step={ranked_step} loss={row['validation_loss']:.8f} -> checkpoint-{ranked_checkpoint_step}")
else:
    ranked_validation_rows = []
    best_validation_row = None
    best_validation_step = None
    best_checkpoint_name = latest_checkpoint_name
    print('No validation rows were logged; defaulting to the latest checkpoint.')
    print('Suggested checkpoint for inference:', best_checkpoint_name)

best_checkpoint_dir = RUN_LOCAL / best_checkpoint_name
print('Best checkpoint directory:', best_checkpoint_dir)
print('Latest checkpoint directory:', latest_checkpoint_dir)
print('Final train loss:', train_rows[-1]['train_loss'])


## Result

This run trains the 5k `BG / HAIR / FACE` target scheme with LR `1e-5`, rank `8`,
LoRA alpha `8`, 200 steps, and checkpoints every 25 steps.
